<a href="https://colab.research.google.com/github/jetnipitbank-maker/thai-bank-sentiment-analysis/blob/main/06_banks_news_labeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# [CELL-00]
!pip install -q transformers[torch] datasets pythainlp tqdm pandas numpy torch

In [ ]:
# [PART 1 - ULTIMATE: รวมไฟล์ + กรองขยะจัดเต็ม + แปลงวันที่ + แสดงสถิติแบงก์]
import pandas as pd
import json
import glob
import os
import re

# =======================================================
# 1. กำหนด Path
# =======================================================
folder_path = r"C:\Users\USER\Desktop\BANKS_News\Final\ForAnalyze"
output_clean_file = r"C:\Users\USER\Desktop\Cleaned_Unlabeled_News.csv"

print(f"🔄 1. เริ่มค้นหาไฟล์ .jsonl ในโฟลเดอร์: '{folder_path}' ...\n")
search_pattern = os.path.join(folder_path, "*.jsonl")
file_list = glob.glob(search_pattern)

if not file_list:
    print(f"❌ ไม่พบไฟล์ .jsonl ในโฟลเดอร์ '{folder_path}'")
else:
    print(f"📂 พบไฟล์ทั้งหมด {len(file_list)} ไฟล์")

all_data = []

# =======================================================
# 2. อ่านไฟล์มารวมกัน
# =======================================================
for file in file_list:
    try:
        with open(file, 'r', encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    all_data.append(json.loads(line.strip()))
    except Exception as e:
        print(f"❌ Error อ่านไฟล์ {os.path.basename(file)}: {e}")

df_all = pd.DataFrame(all_data)
print(f"\n📊 รวมข้อมูลดิบทั้งหมดได้: {len(df_all):,} ข่าว")

# =======================================================
# 3. จัดฟอร์แมตข้อความ (Title + Content)
# =======================================================
df_all['title'] = df_all['title'].fillna("")
df_all['content'] = df_all['content'].fillna("")
df_all['text'] = df_all['title'].astype(str) + " " + df_all['content'].astype(str)
df_all['text'] = df_all['text'].str.strip()

df_all = df_all[df_all['text'] != ""].copy()

# =======================================================
# 🌟 4. คลีนวันที่ (แปลงไทย/พ.ศ. เป็น ค.ศ. และ "ตัดเวลาทิ้ง")
# =======================================================
print("📅 กำลังแปลง Format วันที่ให้เป็นมาตรฐาน (YYYY-MM-DD) และตัดเวลาทิ้ง...")

thai_months = {
    'ม.ค.': '01', 'มกราคม': '01', 'ก.พ.': '02', 'กุมภาพันธ์': '02',
    'มี.ค.': '03', 'มีนาคม': '03', 'เม.ย.': '04', 'เมษายน': '04',
    'พ.ค.': '05', 'พฤษภาคม': '05', 'มิ.ย.': '06', 'มิถุนายน': '06',
    'ก.ค.': '07', 'กรกฎาคม': '07', 'ส.ค.': '08', 'สิงหาคม': '08',
    'ก.ย.': '09', 'กันยายน': '09', 'ต.ค.': '10', 'ตุลาคม': '10',
    'พ.ย.': '11', 'พฤศจิกายน': '11', 'ธ.ค.': '12', 'ธันวาคม': '12'
}

def clean_date_and_remove_time(date_str):
    if pd.isna(date_str): return None
    date_str = str(date_str).strip()

    # 1. ตัด "เวลา" ทิ้ง (เช่น 15:30, 10.00 น., 09:15:00)
    date_str = re.sub(r'\b\d{1,2}:\d{2}(?::\d{2})?\b', '', date_str)
    date_str = re.sub(r'\b\d{1,2}\.\d{2}\s*น\.?', '', date_str)
    date_str = re.sub(r'เวลา|เมื่อ|อัปเดต|ที่ผ่านมา', '', date_str)

    # 2. แทนที่ชื่อเดือนภาษาไทยด้วยตัวเลข
    for th, en in thai_months.items():
        date_str = date_str.replace(th, f"-{en}-")

    # 3. แปลงปี พ.ศ. (25xx) เป็น ค.ศ.
    def convert_be_to_ce(match):
        year_be = int(match.group())
        if 2500 <= year_be <= 2600:
            return str(year_be - 543)
        return str(year_be)

    date_str = re.sub(r'\b(25\d{2})\b', convert_be_to_ce, date_str)

    # 4. ล้างอักขระขยะ
    date_str = re.sub(r'[\s/]+', '-', date_str)
    date_str = re.sub(r'-+', '-', date_str)
    return date_str.strip('-')

df_all['date_cleaned'] = df_all['date'].apply(clean_date_and_remove_time)
df_all['date'] = pd.to_datetime(df_all['date_cleaned'], format='mixed', errors='coerce')
df_all['date'] = df_all['date'].dt.strftime('%Y-%m-%d')
df_all = df_all.drop(columns=['date_cleaned'])

invalid_dates = df_all['date'].isna().sum()
print(f"✅ แปลงวันที่เสร็จสิ้น! (ทิ้งข่าวที่ไม่มีวันที่/วันที่พังไป {invalid_dates:,} ข่าว)")
df_all = df_all.dropna(subset=['date']).copy()

# =======================================================
# 5. ลบข้อมูลซ้ำ (Duplicate)
# =======================================================
subset_cols = ['url', 'related_banks'] if 'related_banks' in df_all.columns else ['url']
df_cleaned = df_all.drop_duplicates(subset=subset_cols).copy()
print(f"🧹 ลบข่าวซ้ำออกไป: {len(df_all) - len(df_cleaned):,} ข่าว")

# =======================================================
# 6. กรองข่าวขยะ (Blacklist Filter)
# =======================================================
blacklist_groups = {
    "ไม่ใช่ธนาคาร (ชื่อคล้าย)": [
        "สายการบินกรุงเทพ", "บางกอกแอร์เวย์ส", "มหาวิทยาลัยกรุงเทพ", "กรุงเทพมหานคร", "กทม.",
        "โรงพยาบาลกรุงเทพ", "กรุงเทพประกัน", "BDMS", "BKI", "BLA", "ทางด่วนกรุงเทพ", "รถไฟฟ้ากรุงเทพ",
        r"(?<!ธนาคาร)กรุงเทพฯ", r"(?<!ธนาคาร)กรุงเทพ"
    ],
    "ศูนย์วิจัย/เศรษฐกิจมหภาค": [
        "ศูนย์วิจัย", "วิจัยกสิกรไทย", "วิจัยกรุงศรี", "ttb analytics", "EIC", "อีไอซี", "วิจัยเศรษฐกิจ", "สภาพัฒน์", "ค่าเงินบาท"
    ],
    "สรุปภาวะตลาด/สถิติรายวัน": [
        "สรุปหุ้น", "ขายชอร์ต", "ชอร์ตเซล", "หุ้นไทยแนวโน้ม", "หุ้นไทยปิด", "หุ้นไทยลุ้น", "ภาวะตลาดหุ้น", "สรุปการซื้อขาย", "NVDR", "บิ๊กล็อต", "Big Lot"
    ],
    "บริษัทในเครือ (บล./บลจ.)": [
        r"บล\.", "บริษัทหลักทรัพย์", r"บลจ\.", "กองทุนรวม"
    ],
    "ข่าว Noise เฉพาะ KTB": [
       "แอกซ่า", "AXA", "ประกันชีวิต", "ประกันภัย", # กรองธุรกิจประกัน
       "บัตรสวัสดิการแห่งรัฐ", "บัตรคนจน", "สิทธิสวัสดิการ", # กรองนโยบายรัฐ
       "แอปเป๋าตัง", "เป๋าตัง", "ทางรัฐ", # กรองแอปฯ ที่ไม่ใช่ Core Banking
       "หุ้นกู้", "พันธบัตร", "เปิดจองซื้อ", "หน่วยละ 1,000", # กรองข่าวขายหุ้นกู้ทั่วไป
       "เราเที่ยวด้วยกัน", "คนละครึ่ง", "เติมเงินหมื่น" # กรองโครงการกระตุ้นเศรษฐกิจ
    ]
}

print("\n🗑️ เริ่มสแกนและเตะข่าวขยะออก...")
total_irrelevant = 0

for group_name, keywords in blacklist_groups.items():
    pattern = '|'.join(keywords)
    mask_title = df_cleaned['title'].str.contains(pattern, case=False, regex=True)
    mask_content = df_cleaned['content'].str.contains(pattern, case=False, regex=True)
    mask_combined = mask_title | mask_content

    drop_count = mask_combined.sum()
    total_irrelevant += drop_count
    print(f"   - เตะหมวด '{group_name}': {drop_count:,} ข่าว")
    df_cleaned = df_cleaned[~mask_combined].copy()

print(f"🔴 รวมข่าวขยะที่ถูกเตะออกทั้งหมด: {total_irrelevant:,} ข่าว")
print(f"✨ ข่าวที่พร้อมส่งเข้า Model Inference: {len(df_cleaned):,} ข่าว")

# =======================================================
# 🌟 6.5 แสดงสถิติข่าวรายธนาคาร (เพิ่มกลับมาแล้วครับ!)
# =======================================================
if 'related_banks' in df_cleaned.columns:
    print("\n🏦 สถิติข่าวรายธนาคาร (พร้อมใช้งาน):")
    bank_counts = df_cleaned['related_banks'].value_counts()
    for bank, count in bank_counts.items():
        print(f"   - {bank}: {count:,} ข่าว")

# =======================================================
# 7. เซฟไฟล์เตรียมส่งให้ Part 2
# =======================================================
desired_cols = ['related_banks', 'title', 'content', 'text', 'date', 'url']
existing_cols = [col for col in desired_cols if col in df_cleaned.columns]
df_cleaned = df_cleaned[existing_cols]

df_cleaned.to_csv(output_clean_file, index=False, encoding='utf-8-sig')

print("\n" + "="*50)
print(f"✅ บันทึกไฟล์สำเร็จ! พร้อมส่งเข้าโมเดลแล้วครับ")
print(f"📁 เปิดดูวันที่ใน Excel ได้ที่: {output_clean_file}")
print("="*50)

In [ ]:
# [PART 2 - FinBERT STANDARD: Inference + Polarity Formula + Threshold]
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer

# =======================================================
# 1. กำหนด Path
# =======================================================
input_clean_file = r"C:\Users\USER\Desktop\Cleaned_Unlabeled_News.csv"
saved_model_path = r"C:\Users\USER\Desktop\Thai_FinBERT_Sentiment_Model"
final_output_file = r"C:\Users\USER\Desktop\Final_Labeled_News_Dataset.csv"

print(f"📥 1. โหลดข้อมูลจาก: {input_clean_file}")
df_clean = pd.read_csv(input_clean_file)
df_clean['text'] = df_clean['text'].astype(str).fillna("")

# =======================================================
# 🤖 2. เตรียม Model และ Tokenizer
# =======================================================
print("🤖 2. โหลดโมเดล FinBERT เข้าสู่ระบบ...")
tokenizer = AutoTokenizer.from_pretrained(saved_model_path)
model = AutoModelForSequenceClassification.from_pretrained(saved_model_path)

hf_predict_dataset = Dataset.from_pandas(df_clean[['text']])
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=400)

print("📝 กำลัง Tokenize ข้อมูล...")
tokenized_datasets = hf_predict_dataset.map(tokenize_function, batched=True)
tokenized_datasets = tokenized_datasets.remove_columns(["text"])
tokenized_datasets.set_format("torch")

# =======================================================
# 🔍 3. ทำนายผลและใช้สูตรคณิตศาสตร์ FinBERT
# =======================================================
print("\n🔍 3. ให้โมเดลทำนายและคำนวณ Polarity Score...")
trainer = Trainer(model=model)
predictions = trainer.predict(tokenized_datasets)

# แปลง Logits เป็นค่าความน่าจะเป็น (0.0 - 1.0)
logits = torch.tensor(predictions.predictions)
probs = F.softmax(logits, dim=-1).numpy()

# ดึงค่าความน่าจะเป็นแต่ละคลาส (อิงจาก Label: 0=Neg, 1=Neu, 2=Pos)
p_neg = probs[:, 0]
p_neu = probs[:, 1]
p_pos = probs[:, 2]

# 🌟 ใช้สูตร FinBERT: Polarity Score = P(Positive) - P(Negative)
polarity_scores = p_pos - p_neg
confidence_scores = np.max(probs, axis=-1)
predicted_classes = np.argmax(probs, axis=-1)

# 🌟 ตั้งค่า Threshold กรองความลังเล (Noise Filtering)
THRESHOLD = 0.60

sentiment_labels = []
for i in range(len(probs)):
    # ถ้าโมเดลมีความมั่นใจไม่ถึง 60% ให้มองว่าเป็นข่าวกลางๆ (Neutral) ทันที
    if confidence_scores[i] < THRESHOLD:
        polarity_scores[i] = 0.0  # ล้างคะแนนให้เป็น 0
        sentiment_labels.append('Neutral')
    else:
        # ถ้ามั่นใจเกิน 60% ให้ระบุ Label ตามจริง
        if predicted_classes[i] == 0:
            sentiment_labels.append('Negative')
        elif predicted_classes[i] == 2:
            sentiment_labels.append('Positive')
        else:
            sentiment_labels.append('Neutral')

# =======================================================
# 💾 4. บันทึกผลลัพธ์
# =======================================================
df_clean['polarity_score'] = np.round(polarity_scores, 4)
df_clean['confidence_score'] = np.round(confidence_scores, 4)
df_clean['sentiment_label'] = sentiment_labels

# จัดคอลัมน์ให้สวยงาม
final_cols = ['date', 'related_banks', 'title', 'content', 'url', 'sentiment_label', 'polarity_score', 'confidence_score']
final_cols = [col for col in final_cols if col in df_clean.columns]
df_final = df_clean[final_cols]

df_final.to_csv(final_output_file, index=False, encoding='utf-8-sig')
print(f"✅ บันทึกไฟล์ Final เรียบร้อยแล้วที่: {final_output_file}")
display(df_final[['title', 'sentiment_label', 'polarity_score', 'confidence_score']].head(5))

In [ ]:
import pandas as pd
import os

# 1. กำหนด Path ไฟล์ต้นฉบับและโฟลเดอร์ปลายทาง
input_file = r"C:\Users\USER\Desktop\Final_Labeled_News_Dataset.csv"
output_folder = r"C:\Users\USER\Desktop\Banks_Separated_News" # โฟลเดอร์สำหรับเก็บไฟล์ย่อย

# สร้างโฟลเดอร์ถ้ายังไม่มี
if not os.path.exists(output_folder):
    os.makedirs(output_folder)
    print(f"📁 สร้างโฟลเดอร์ใหม่สำหรับเก็บไฟล์: {output_folder}")

print(f"📥 กำลังโหลดข้อมูลจาก: {input_file}")
df = pd.read_csv(input_file)

# 2. บังคับให้คอลัมน์ date เป็น Datetime เพื่อให้เรียงลำดับได้อย่างถูกต้อง
df['date'] = pd.to_datetime(df['date'])

# รายชื่อ 6 ธนาคารเป้าหมาย
target_banks = ['KBANK', 'SCB', 'BBL', 'KTB', 'BAY', 'TTB']

print("\n🔄 เริ่มกระบวนการแยกไฟล์และเรียงวันที่...")

# 3. วนลูปแยกข้อมูลทีละธนาคาร
for bank in target_banks:
    # กรองเอาเฉพาะธนาคารที่กำลังวนลูป
    # ใช้ str.contains เผื่อกรณีที่มีชื่อธนาคารอื่นติดมาด้วย
    df_bank = df[df['related_banks'].astype(str).str.contains(bank, na=False)].copy()

    if len(df_bank) > 0:
        # 🌟 เรียงลำดับตามวันที่ (จากเก่าไปใหม่)
        df_bank = df_bank.sort_values(by='date', ascending=True).reset_index(drop=True)

        # ปรับฟอร์แมตวันที่กลับเป็นข้อความ YYYY-MM-DD ให้สวยงามก่อนเซฟ
        df_bank['date'] = df_bank['date'].dt.strftime('%Y-%m-%d')

        # ตั้งชื่อไฟล์ใหม่ตามชื่อธนาคาร
        output_filename = os.path.join(output_folder, f"{bank}_Labeled_News.csv")

        # บันทึกเป็น CSV
        df_bank.to_csv(output_filename, index=False, encoding='utf-8-sig')
        print(f"   ✅ {bank}: เซฟเรียบร้อย ({len(df_bank):,} ข่าว) -> {os.path.basename(output_filename)}")
    else:
        print(f"   ⚠️ {bank}: ไม่พบข้อมูลในระบบ")

print("\n" + "="*50)
print(f"🎉 แยกไฟล์เสร็จสมบูรณ์ทั้ง 6 ธนาคาร!")
print(f"📁 คุณสามารถเข้าไปดูไฟล์ทั้งหมดได้ที่: {output_folder}")
print("="*50)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML

# 🎨 ตั้งค่าฟอนต์ภาษาไทยและสไตล์กราฟ
plt.rcParams['font.family'] = 'Tahoma' # เปลี่ยนเป็นฟอนต์ที่คุณใช้ในเครื่อง
sns.set_theme(style="whitegrid", rc={"font.family": "Tahoma"})

# ==========================================
# 1. โหลดและเตรียมข้อมูล
# ==========================================
file_path = r"C:\Users\USER\Desktop\Final_Labeled_News_Dataset.csv"
try:
    df = pd.read_csv(file_path)
    df['date'] = pd.to_datetime(df['date'])
    df['year_month'] = df['date'].dt.to_period('M').astype(str)
    print("✅ โหลดข้อมูลสำเร็จ!")
    print(f"จำนวนข่าวทั้งหมด: {len(df):,} ข่าว\n")
except Exception as e:
    print(f"❌ โหลดข้อมูลไม่สำเร็จ: {e}")

# ตั้งค่าสี Theme สำหรับตาราง
bg_color, head_bg_color = '#f9fafb', '#1f2937'
text_color, head_text_color = '#111827', '#ffffff'
line_color = '#e5e7eb'

def apply_table_style(styler, title):
    return (styler.set_caption(f"<b style='font-size:16px; color:#1f2937;'>{title}</b>")
            .set_table_styles([
                {'selector': 'table', 'props': [('border-collapse', 'collapse'), ('margin-bottom', '20px'), ('width', '100%'), ('font-family', 'Segoe UI, sans-serif')]},
                {'selector': 'caption', 'props': [('text-align', 'left'), ('padding-bottom', '10px')]},
                {'selector': 'th', 'props': [('background-color', head_bg_color), ('color', head_text_color), ('font-weight', 'bold'), ('padding', '12px'), ('text-align', 'center'), ('border', f'1px solid {line_color}')]},
                {'selector': 'td', 'props': [('background-color', bg_color), ('color', text_color), ('padding', '10px'), ('text-align', 'center'), ('border', f'1px solid {line_color}')]},
                {'selector': 'th:first-child, td:first-child', 'props': [('text-align', 'left'), ('font-weight', 'bold')]}
            ]))

# ==========================================
# 📊 2. เจาะลึกความมั่นใจและคะแนนอารมณ์รายหมวด (Confidence by Sentiment)
# ==========================================

# 1. หาค่าเฉลี่ยและค่าต่ำสุดของคะแนนความมั่นใจ/อารมณ์ ก่อน (ไม่เอามารวมกับการนับ)
sentiment_detail = df.groupby('sentiment_label').agg(
    ความมั่นใจเฉลี่ย=('confidence_score', 'mean'),
    ความมั่นใจ_Min=('confidence_score', 'min'),
    คะแนนอารมณ์เฉลี่ย=('polarity_score', 'mean')
)

# 2. นับจำนวนข่าวและคำนวณสัดส่วนเปอร์เซ็นต์แยกต่างหาก (วิธีนี้ Pandas ไม่ Error แน่นอน)
sentiment_detail['จำนวนข่าว'] = df['sentiment_label'].value_counts()
sentiment_detail['สัดส่วนเปอร์เซ็นต์'] = (sentiment_detail['จำนวนข่าว'] / len(df)) * 100

# 3. จัดเรียงคอลัมน์ให้สวยงามและเรียงตามจำนวนข่าวจากมากไปน้อย
sentiment_detail = sentiment_detail[['จำนวนข่าว', 'สัดส่วนเปอร์เซ็นต์', 'ความมั่นใจเฉลี่ย', 'ความมั่นใจ_Min', 'คะแนนอารมณ์เฉลี่ย']]
sentiment_detail = sentiment_detail.sort_values('จำนวนข่าว', ascending=False)
sentiment_detail.index.name = 'ขั้วอารมณ์'

# 4. แสดงผลตารางแบบไล่เฉดสี
styled_sentiment = apply_table_style(
    sentiment_detail.style.format({
        'จำนวนข่าว': "{:,.0f}", 'สัดส่วนเปอร์เซ็นต์': "{:.2f}%",
        'ความมั่นใจเฉลี่ย': "{:.4f}", 'ความมั่นใจ_Min': "{:.4f}", 'คะแนนอารมณ์เฉลี่ย': "{:.4f}"
    }).background_gradient(cmap='Greens', subset=['ความมั่นใจเฉลี่ย']),
    "ตารางที่ 1: การกระจายตัวของข่าวสารและระดับความมั่นใจของแบบจำลองแยกตามขั้วอารมณ์"
)
display(styled_sentiment)

# ==========================================
# 🏦 3. เจาะลึกอารมณ์ข่าวรายธนาคาร (Bank-Specific Sentiment & Polarity)
# ==========================================
# ใช้วิธี Crosstab เพื่อนับจำนวนข่าวแยกตามธนาคารและอารมณ์แบบอัตโนมัติ (ไม่ต้องฮาร์ดโค้ดชื่อ Label)
bank_detail = pd.crosstab(df['related_banks'], df['sentiment_label'])

# เพิ่มคอลัมน์ "จำนวนข่าวรวม" และ "ค่าเฉลี่ย Polarity"
bank_detail['จำนวนข่าวรวม'] = bank_detail.sum(axis=1)
bank_detail['Polarity_Score_เฉลี่ย'] = df.groupby('related_banks')['polarity_score'].mean()

# เรียงจากธนาคารที่มีข่าวเยอะสุดไปน้อยสุด
bank_detail = bank_detail.sort_values('จำนวนข่าวรวม', ascending=False)
bank_detail.index.name = 'ธนาคาร'

# ย้ายคอลัมน์ จำนวนข่าวรวม มาไว้หน้าสุดให้ดูง่าย
cols = ['จำนวนข่าวรวม'] + [c for c in bank_detail.columns if c not in ['จำนวนข่าวรวม', 'Polarity_Score_เฉลี่ย']] + ['Polarity_Score_เฉลี่ย']
bank_detail = bank_detail[cols]

# ตั้งค่า Format ให้ทศนิยม 4 ตำแหน่งเฉพาะคอลัมน์ Polarity นอกนั้นเป็นจำนวนเต็ม
format_dict = {col: "{:,.0f}" for col in bank_detail.columns if col != 'Polarity_Score_เฉลี่ย'}
format_dict['Polarity_Score_เฉลี่ย'] = "{:.4f}"

styled_bank = apply_table_style(
    bank_detail.style.format(format_dict).background_gradient(cmap='coolwarm', subset=['Polarity_Score_เฉลี่ย']),
    "ตารางที่ 2: ปริมาณข่าวสารและค่าเฉลี่ยคะแนนอารมณ์ (Polarity Score) แยกรายธนาคาร"
)
display(styled_bank)

# ==========================================
# 📈 4. วาดกราฟวิเคราะห์เชิงลึก (Advanced Visualizations)
# ==========================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# กราฟ 4.1: Boxplot ดูการกระจายตัวของความมั่นใจ (Confidence Score Distribution)
sns.boxplot(data=df, x='sentiment_label', y='confidence_score', palette='Set2', ax=axes[0])
axes[0].set_title('การกระจายตัวของความมั่นใจ (Confidence Score) แยกตามขั้วอารมณ์', fontsize=14, fontweight='bold')
axes[0].set_xlabel('ขั้วอารมณ์', fontsize=12)
axes[0].set_ylabel('Confidence Score', fontsize=12)

# กราฟ 4.2: Stacked Bar Chart แนวโน้มข่าวรายเดือน
monthly_counts = df.groupby(['year_month', 'sentiment_label']).size().unstack(fill_value=0)
monthly_counts.plot(kind='bar', stacked=True, colormap='viridis', ax=axes[1])

axes[1].set_title('สัดส่วนขั้วอารมณ์ข่าวสารในแต่ละเดือน (Monthly Sentiment Breakdown)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('เดือน-ปี', fontsize=12)
axes[1].set_ylabel('จำนวนข่าว (รายการ)', fontsize=12)
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend(title='Sentiment')

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML

# 🎨 ตั้งค่าฟอนต์ภาษาไทยและสไตล์กราฟ
plt.rcParams['font.family'] = 'Tahoma' # เปลี่ยนเป็นฟอนต์ที่แสดงผลไทยได้ดีในเครื่องคุณ
sns.set_theme(style="whitegrid", rc={"font.family": "Tahoma"})

# ==========================================
# 1. โหลดและเตรียมข้อมูล
# ==========================================
file_path = r"C:\Users\USER\Desktop\Final_Labeled_News_Dataset.csv"
try:
    df = pd.read_csv(file_path)
    df['date'] = pd.to_datetime(df['date'])
    df['year_month'] = df['date'].dt.to_period('M').astype(str)
    print("✅ โหลดข้อมูลสำเร็จ!")
    print(f"จำนวนข่าวทั้งหมด: {len(df):,} ข่าว\n")
except Exception as e:
    print(f"❌ โหลดข้อมูลไม่สำเร็จ: {e}")

# ตั้งค่าสี Theme สำหรับตารางสไตล์วิชาการ
bg_color, head_bg_color = '#f9fafb', '#1f2937'
text_color, head_text_color = '#111827', '#ffffff'
line_color = '#dee2e6'

# ฟังก์ชันจัดการสไตล์ตารางและเน้นตัวหนาที่บรรทัดสุดท้าย (Total Row)
def apply_table_style(styler, title):
    return (styler.set_caption(f"<b style='font-size:16px; color:#1f2937;'>{title}</b>")
            .set_table_styles([
                {'selector': 'table', 'props': [('border-collapse', 'collapse'), ('margin-bottom', '25px'), ('width', '100%'), ('font-family', 'Segoe UI, sans-serif')]},
                {'selector': 'caption', 'props': [('text-align', 'left'), ('padding-bottom', '10px')]},
                {'selector': 'th', 'props': [('background-color', head_bg_color), ('color', head_text_color), ('font-weight', 'bold'), ('padding', '12px'), ('text-align', 'center'), ('border', f'1px solid {line_color}')]},
                {'selector': 'td', 'props': [('background-color', bg_color), ('color', text_color), ('padding', '10px'), ('text-align', 'center'), ('border', f'1px solid {line_color}')]},
                {'selector': 'th:first-child, td:first-child', 'props': [('text-align', 'left'), ('font-weight', 'bold')]},
                # 🌟 จัดสไตล์บรรทัดสุดท้าย (Total Row) ให้เป็นตัวหนา และมีเส้นคั่นด้านบนหนาขึ้น
                {'selector': 'tbody tr:last-child td', 'props': [('font-weight', 'bold'), ('background-color', '#f3f4f6'), ('border-top', f'2.5px double #1f2937')]}
            ]))

# ==========================================
# 📊 2. ตารางที่ 1: แยกตามขั้วอารมณ์ + เพิ่มบรรทัดรวม (Total Row)
# ==========================================
print("📊 1. ภาพรวมอารมณ์ข่าวสาร (Overall Sentiment Distribution)")

# คำนวณค่ารายกลุ่ม
sentiment_detail = df.groupby('sentiment_label').agg(
    ความมั่นใจเฉลี่ย=('confidence_score', 'mean'),
    ความมั่นใจ_Min=('confidence_score', 'min'),
    คะแนนอารมณ์เฉลี่ย=('polarity_score', 'mean')
)
sentiment_detail['จำนวนข่าว'] = df['sentiment_label'].value_counts()
sentiment_detail['สัดส่วนเปอร์เซ็นต์'] = (sentiment_detail['จำนวนข่าว'] / len(df)) * 100
sentiment_detail = sentiment_detail[['จำนวนข่าว', 'สัดส่วนเปอร์เซ็นต์', 'ความมั่นใจเฉลี่ย', 'ความมั่นใจ_Min', 'คะแนนอารมณ์เฉลี่ย']]
sentiment_detail = sentiment_detail.sort_values('จำนวนข่าว', ascending=False)

# คำนวณบรรทัดรวมสำหรับตารางที่ 1 (ใช้ข้อมูลประชากรทั้งหมด)
total_row_1 = pd.DataFrame([{
    'จำนวนข่าว': len(df),
    'สัดส่วนเปอร์เซ็นต์': 100.00,
    'ความมั่นใจเฉลี่ย': df['confidence_score'].mean(),
    'ความมั่นใจ_Min': df['confidence_score'].min(),
    'คะแนนอารมณ์เฉลี่ย': df['polarity_score'].mean()
}], index=['รวมทั้งหมด'])

# ต่อตารางหลักเข้ากับบรรทัดรวม
sentiment_table_final = pd.concat([sentiment_detail, total_row_1])
sentiment_table_final.index.name = 'ขั้วอารมณ์'

styled_sentiment = apply_table_style(
    sentiment_table_final.style.format({
        'จำนวนข่าว': "{:,.0f}", 'สัดส่วนเปอร์เซ็นต์': "{:.2f}%",
        'ความมั่นใจเฉลี่ย': "{:.4f}", 'ความมั่นใจ_Min': "{:.4f}", 'คะแนนอารมณ์เฉลี่ย': "{:.4f}"
    }),
    "ตารางที่ 1: การกระจายตัวของข่าวสารและระดับความมั่นใจของแบบจำลองแยกตามขั้วอารมณ์ (พร้อมบรรทัดรวม)"
)
display(styled_sentiment)


# ==========================================
# 🏦 3. ตารางที่ 2: แยกรายธนาคาร + เพิ่มบรรทัดรวม (Total Row)
# ==========================================
print("\n🏦 2. ปริมาณข่าวและอารมณ์แยกตามธนาคาร (Bank Breakdown)")

# สร้าง Cross Tabulation นับจำนวนข่าวดี/กลาง/ร้าย รายธนาคาร
bank_detail = pd.crosstab(df['related_banks'], df['sentiment_label'])
bank_detail['จำนวนข่าวรวม'] = bank_detail.sum(axis=1)
bank_detail['Polarity_Score_เฉลี่ย'] = df.groupby('related_banks')['polarity_score'].mean()
bank_detail = bank_detail.sort_values('จำนวนข่าวรวม', ascending=False)

# จัดเรียงลำดับคอลัมน์ให้เหมาะสม
cols = ['จำนวนข่าวรวม'] + [c for c in bank_detail.columns if c not in ['จำนวนข่าวรวม', 'Polarity_Score_เฉลี่ย']] + ['Polarity_Score_เฉลี่ย']
bank_detail = bank_detail[cols]

# คำนวณบรรทัดรวมสำหรับตารางที่ 2
total_values = {col: bank_detail[col].sum() for col in bank_detail.columns if col != 'Polarity_Score_เฉลี่ย'}
total_values['Polarity_Score_เฉลี่ย'] = df['polarity_score'].mean() # ค่าเฉลี่ยอารมณ์รวมของตลาดทั้งหมด

total_row_2 = pd.DataFrame([total_values], index=['รวมทั้งหมด'])

# ต่อตารางหลักเข้ากับบรรทัดรวม
bank_table_final = pd.concat([bank_detail, total_row_2])
bank_table_final.index.name = 'ธนาคาร'

# ตั้งค่า Format การแสดงผลตัวเลข
format_dict = {col: "{:,.0f}" for col in bank_table_final.columns if col != 'Polarity_Score_เฉลี่ย'}
format_dict['Polarity_Score_เฉลี่ย'] = "{:.4f}"

styled_bank = apply_table_style(
    bank_table_final.style.format(format_dict),
    "ตารางที่ 2: ปริมาณข่าวสารและค่าเฉลี่ยคะแนนอารมณ์ (Polarity Score) แยกรายธนาคาร (พร้อมบรรทัดรวม)"
)
display(styled_bank)